# Activity 3 – Data Visualisation
**Course:** Advanced Programming – Week 6  
**Author:** Sebastian Diaz  

Uses `CollegeGrades1.csv` (the combined dataset referenced in the activity).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
import tkinter as tk

# Re-use the loader from Activity 1
def load_college_grades(filepath, grade_order, period_labels, data_rows):
    raw = pd.read_csv(filepath, header=None, dtype=str)
    def ffill_row(series):
        result, current = [], ''
        for v in series:
            val = str(v).strip()
            if val and val != 'nan':
                current = val
            result.append(current)
        return result
    colleges = ffill_row(raw.iloc[0, 2:])
    subjects = ffill_row(raw.iloc[2, 2:])
    grades   = [str(v).strip() for v in raw.iloc[4, 2:]]
    col_idx  = pd.MultiIndex.from_arrays([colleges, subjects, grades],
                                          names=['College','Subject','Grade'])
    data     = raw.iloc[data_rows, 2:].values.astype(int)
    periods  = [p for p, g in period_labels]
    genders  = [g for p, g in period_labels]
    row_idx  = pd.MultiIndex.from_arrays([periods, genders], names=['Period','Gender'])
    return pd.DataFrame(data, index=row_idx, columns=col_idx)

df = load_college_grades(
    'CollegeGrades1.csv',
    grade_order=['F','P','M','D'],
    period_labels=[
        ('Year 1','M'),('Year 1','F'),('Year 2','M'),('Year 2','F'),
        ('Evening','M'),('Evening','F')
    ],
    data_rows=[6,8,10,12,14,16]
)
# Fix known typos
spelling = {'Enginering':'Engineering','Psycology':'Psychology',
            'Chemisrty':'Chemistry','Scarborugh':'Scarborough'}
new_cols = df.columns.to_frame()
for level in ['College','Subject']:
    new_cols[level] = new_cols[level].replace(spelling)
df = df.set_axis(pd.MultiIndex.from_frame(new_cols), axis=1)
print("Dataset loaded. Shape:", df.shape)

---
## Exercise 1 – Matplotlib + Tkinter Framework with Swappable Graphs

A `ChartFrame` class holds a Matplotlib figure embedded in tkinter.
Calling `swap_chart(kind)` replaces the plot without recreating the window.

In [ ]:
class ChartFrame:
    """
    Reusable tkinter frame containing an embedded Matplotlib figure.
    Call swap_chart(kind) to replace the current plot with 'bar', 'line',
    'hist', or 'scatter' using the same data.
    """

    def __init__(self, root, x_data, y_data, title='Chart', xlabel='', ylabel=''):
        self.root   = root
        self.x_data = x_data
        self.y_data = y_data
        self.title  = title
        self.xlabel = xlabel
        self.ylabel = ylabel

        # Create figure and embed
        self.fig = Figure(figsize=(8, 5), dpi=100)
        self.ax  = self.fig.add_subplot(111)

        self.canvas = FigureCanvasTkAgg(self.fig, master=root)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

        toolbar = NavigationToolbar2Tk(self.canvas, root)
        toolbar.update()

        # Button row for swapping
        btn_frame = tk.Frame(root)
        btn_frame.pack(fill=tk.X)
        for kind in ['bar', 'line', 'hist', 'scatter']:
            tk.Button(btn_frame, text=kind.capitalize(),
                      command=lambda k=kind: self.swap_chart(k)).pack(side=tk.LEFT, padx=4, pady=4)

        self.swap_chart('bar')   # default

    def swap_chart(self, kind):
        """Clear axes and redraw with the requested chart type."""
        self.ax.clear()
        x, y = self.x_data, self.y_data
        x_pos = range(len(x))

        if kind == 'bar':
            self.ax.bar(x_pos, y, color='steelblue', edgecolor='white')
            self.ax.set_xticks(x_pos)
            self.ax.set_xticklabels(x, rotation=45, ha='right')

        elif kind == 'line':
            self.ax.plot(x_pos, y, marker='o', color='darkorange', linewidth=2)
            self.ax.set_xticks(x_pos)
            self.ax.set_xticklabels(x, rotation=45, ha='right')

        elif kind == 'hist':
            self.ax.hist(y, bins=10, color='seagreen', edgecolor='white')

        elif kind == 'scatter':
            self.ax.scatter(x_pos, y, color='crimson', s=80, zorder=3)
            self.ax.set_xticks(x_pos)
            self.ax.set_xticklabels(x, rotation=45, ha='right')

        self.ax.set_title(f'{self.title} ({kind})')
        self.ax.set_xlabel(self.xlabel)
        self.ax.set_ylabel(self.ylabel)
        self.ax.grid(True, linestyle='--', alpha=0.4)
        self.fig.tight_layout()
        self.canvas.draw()


# --- Demo run ---
# Prepare some data: total students per subject across all colleges
subject_totals = df.groupby(level='Subject', axis=1).sum().sum()
subjects = subject_totals.index.tolist()
totals   = subject_totals.values.tolist()

print("ChartFrame framework defined.")
print("To launch the GUI, run the block below (not in Colab — run as a .py file or in Jupyter on a local machine).")

In [ ]:
# Launch the demo window (local Jupyter / .py only — not Colab)
# Uncomment to run:
# root = tk.Tk()
# root.title('Week 6 – Swappable Charts Demo')
# ChartFrame(root, subjects, totals, title='Total Students', xlabel='Subject', ylabel='Count')
# root.mainloop()

---
## Exercise 2 – Analysis Questions with Visualisations

### Q1: Overall success rate (Pass or above in Year 2) per subject

In [ ]:
# Success = grade P, M, or D in Year 2 (not F)
yr2 = df.xs('Year 2', level='Period')

# Sum across genders and colleges → subject × grade
yr2_by_subject = yr2.groupby(level=['Subject','Grade'], axis=1).sum().sum()
yr2_df = yr2_by_subject.unstack(level='Grade')
yr2_df = yr2_df[['F','P','M','D']]   # canonical order

yr2_df['Total']   = yr2_df.sum(axis=1)
yr2_df['Success'] = yr2_df['P'] + yr2_df['M'] + yr2_df['D']
yr2_df['Success%'] = (yr2_df['Success'] / yr2_df['Total'] * 100).round(1)

print("Year 2 Success Rate (Pass or above) by Subject:")
print(yr2_df[['F','P','M','D','Total','Success%']].sort_values('Success%', ascending=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d73027','#fee090','#91cf60','#1a9850']  # red=fail, yellow=pass, greens=merit/dist

bottom = np.zeros(len(yr2_df))
for grade, color in zip(['F','P','M','D'], colors):
    vals = yr2_df[grade].values
    ax.bar(yr2_df.index, vals, bottom=bottom, label=grade, color=color, edgecolor='white')
    bottom += vals

ax.set_title('Year 2 Grade Distribution by Subject\n(stacked: F=Fail, P=Pass, M=Merit, D=Distinction)',
             fontsize=12)
ax.set_xlabel('Subject')
ax.set_ylabel('Number of Students')
ax.legend(title='Grade', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('q1_year2_success_rate.png', dpi=120, bbox_inches='tight')
plt.show()

### Q2: Year 1 vs Year 2 performance comparison per college

In [ ]:
colleges = df.columns.get_level_values('College').unique().tolist()

fig, axes = plt.subplots(1, len(colleges), figsize=(16, 5), sharey=False)
grade_order = ['F', 'P', 'M', 'D']
colors = {'Year 1': '#4393c3', 'Year 2': '#d6604d'}

for ax, college in zip(axes, colleges):
    # All subjects in this college, sum across genders, compare Year 1 vs Year 2
    coll_data = df[college].groupby(level='Grade', axis=1).sum()

    yr1_totals = coll_data.xs('Year 1', level='Period').sum()
    yr2_totals = coll_data.xs('Year 2', level='Period').sum()

    x = np.arange(len(grade_order))
    width = 0.35

    ax.bar(x - width/2, [yr1_totals.get(g, 0) for g in grade_order],
           width, label='Year 1', color=colors['Year 1'], edgecolor='white')
    ax.bar(x + width/2, [yr2_totals.get(g, 0) for g in grade_order],
           width, label='Year 2', color=colors['Year 2'], edgecolor='white')

    ax.set_title(college)
    ax.set_xticks(x)
    ax.set_xticklabels(grade_order)
    ax.set_xlabel('Grade')
    ax.set_ylabel('Students')
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

fig.suptitle('Year 1 vs Year 2 Performance by College', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q2_yr1_vs_yr2_by_college.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Exercise 3 – Justification of Visualisation Choice (~200 words)

For Q1 (success rate by subject), I chose a **stacked bar chart**. The data is categorical (subjects) with a composition breakdown (F/P/M/D grade distribution), making stacking appropriate: it shows both the total number of students and the proportional contribution of each grade band in a single view. A grouped bar chart would have required four bars per subject, making side-by-side comparison of grade bands visually cluttered across nine subjects. A pie chart per subject would require nine separate figures and would not allow comparison across subjects.

**Limitation**: Stacked bars make it difficult to compare non-baseline segments (e.g., comparing Merit counts across subjects is harder than comparing Fail counts, which sit on the baseline). A 100% stacked bar normalised to percentages would improve proportion comparison, but would lose the information about absolute student counts.

For Q2 (Year 1 vs Year 2 per college), I chose a **grouped bar chart** with one panel per college. This directly answers the comparison question — for each grade category, you can immediately see whether Year 2 produces more or fewer students in that band compared to Year 1. The `sharey=False` option lets each college's y-scale reflect its own student volume, which is appropriate since the colleges are being compared individually, not as a single aggregate.

Alternative viable approaches: a **heatmap** of pass rates would be more compact but would lose the grade composition detail. A **line graph** would imply continuous progression between grades, which is not meaningful for discrete grade categories.

---
## Exercise 4 – Embed Year 1 vs Year 2 Chart in tkinter GUI

In [ ]:
def build_yr1_vs_yr2_figure(df, college):
    """Build and return a Matplotlib Figure for Year 1 vs Year 2 comparison."""
    fig, ax = plt.subplots(figsize=(7, 4))
    grade_order = ['F', 'P', 'M', 'D']

    coll_data  = df[college].groupby(level='Grade', axis=1).sum()
    yr1_totals = coll_data.xs('Year 1', level='Period').sum()
    yr2_totals = coll_data.xs('Year 2', level='Period').sum()

    x = np.arange(len(grade_order))
    w = 0.35
    ax.bar(x - w/2, [yr1_totals.get(g, 0) for g in grade_order], w,
           label='Year 1', color='#4393c3', edgecolor='white')
    ax.bar(x + w/2, [yr2_totals.get(g, 0) for g in grade_order], w,
           label='Year 2', color='#d6604d', edgecolor='white')
    ax.set_title(f'{college} — Year 1 vs Year 2')
    ax.set_xticks(x)
    ax.set_xticklabels(grade_order)
    ax.set_xlabel('Grade')
    ax.set_ylabel('Students')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    fig.tight_layout()
    return fig


def launch_gui(df):
    """Launch a tkinter window with a college selector and embedded chart."""
    colleges = df.columns.get_level_values('College').unique().tolist()

    root = tk.Tk()
    root.title('College Grades — Year 1 vs Year 2')
    root.geometry('750x520')

    # --- Controls ---
    ctrl = tk.Frame(root, bd=1, relief=tk.SUNKEN)
    ctrl.pack(fill=tk.X, padx=8, pady=4)

    tk.Label(ctrl, text='College:', font=('Arial', 11)).pack(side=tk.LEFT, padx=6)
    selected = tk.StringVar(value=colleges[0])
    dropdown = tk.OptionMenu(ctrl, selected, *colleges)
    dropdown.pack(side=tk.LEFT)

    # --- Chart area ---
    chart_frame = tk.Frame(root)
    chart_frame.pack(fill=tk.BOTH, expand=True, padx=8, pady=4)

    # Initial figure
    fig = build_yr1_vs_yr2_figure(df, colleges[0])
    canvas = FigureCanvasTkAgg(fig, master=chart_frame)
    canvas.draw()
    canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    toolbar_frame = tk.Frame(root)
    toolbar_frame.pack(fill=tk.X)
    toolbar = NavigationToolbar2Tk(canvas, toolbar_frame)
    toolbar.update()

    # --- Update on selection ---
    def update(*_):
        college = selected.get()
        new_fig = build_yr1_vs_yr2_figure(df, college)
        # Replace figure in canvas
        canvas.figure = new_fig
        canvas.draw()

    selected.trace_add('write', update)

    root.mainloop()


print("GUI function defined.")
print("To launch: uncomment the line below and run as a .py file or local Jupyter notebook.")
# launch_gui(df)